In [32]:
import json
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.multitest as multitest

def load_json_file(file_path):
    """
    Load a JSON file and return its content as a Python object.

    :param file_path: Path to the JSON file.
    :return: Python object representing the JSON data.
    """
    with open(file_path, 'r') as file:
        data = json.load(file)
    return data

def save_json_file(data, file_path):
    """
    Save a Python object as a JSON file.

    :param data: Python object to be saved as JSON.
    :param file_path: Path where the JSON file will be saved.
    """
    with open(file_path, 'w') as file:
        json.dump(data, file, indent=4)

def print_json_structure(data):
    """
    Pretty print a Python object as JSON.

    :param data: Python object to be printed as JSON.
    """
    for dataset_key, dataset_value in data.items():
        print(f"Dataset: {dataset_key}")
        print(f"  Methods: {', '.join(dataset_value.keys())}")  # Print the remaining method keys
        print(f"  Metrics: {', '.join(dataset_value[next(iter(dataset_value))].keys())}")  # Print the metrics keys from the first method
        print()

# drop agentsimulator and unused metrics

In [33]:
summary = load_json_file('results/summary.json')

for dataset_key, dataset_value in summary.items():
    print(f"Dataset: {dataset_key}")

    del dataset_value['agentsimulator']  # Remove 'agentsimulator' key if it exists

    print(f"  Methods: {', '.join(dataset_value.keys())}")  # Print the remaining method keys
    print(f"  Metrics: {', '.join(dataset_value[next(iter(dataset_value))].keys())}")  # Print the metrics keys from the first method
    print()
    
    summary[dataset_key] = dataset_value  # Update the summary with the modified dataset_value

save_json_file(summary, 'results/summary_modified.json')  # Save the modified summary back to the JSON file


Dataset: BPI_Challenge_2012_W_Two_TS
  Methods: prosit, simod, rims, dsim
  Metrics: transition_tree, resource_tree, 2gd, ctd, car, arrival_tree, waiting_tree, execution_tree, etd_entropy, 2rgd

Dataset: BPI_Challenge_2017_W_Two_TS
  Methods: prosit, simod, rims, dsim
  Metrics: transition_tree, resource_tree, 2gd, ctd, car, arrival_tree, waiting_tree, execution_tree, etd_entropy, 2rgd

Dataset: ConsultaDataMining201618
  Methods: prosit, simod, rims, dsim
  Metrics: transition_tree, resource_tree, 2gd, ctd, car, arrival_tree, waiting_tree, execution_tree, etd_entropy, 2rgd

Dataset: cvs_pharmacy
  Methods: prosit, simod, rims, dsim
  Metrics: car, 2gd, transition_tree, resource_tree, waiting_tree, arrival_tree, etd_entropy, ctd, execution_tree, 2rgd

Dataset: confidential_2000
  Methods: prosit, simod, rims, dsim
  Metrics: car, 2gd, transition_tree, resource_tree, waiting_tree, arrival_tree, etd_entropy, ctd, execution_tree, 2rgd



# convert table json to same json format

In [3]:
TABLES = {'2gd': {'BPIC12W': {'ProSiT': (0.398, 0.004),
                     'AgentSim': (0.456, 0.006),
                     'SIMOD': (0.525, 0.006),
                     'RIMS': (0.525, 0.006),
                     'DSIM': (0.51, 0.007)},
         'BPIC17W': {'ProSiT': (0.216, 0.003),
                     'AgentSim': (0.258, 0.001),
                     'SIMOD': (0.424, 0.003),
                     'RIMS': (0.479, 0.003),
                     'DSIM': (0.549, 0.002)},
         'P2P-1000': {'ProSiT': (0.116, 0.01),
                      'SIMOD': (0.189, 0.009),
                      'RIMS': (0.18, 0.008),
                      'DSIM': (0.184, 0.007)},
         'P2P-2000': {'ProSiT': (0.106, 0.011),
                      'SIMOD': (0.187, 0.008),
                      'RIMS': (0.195, 0.01),
                      'DSIM': (0.21, 0.012)},
         'ACR': {'ProSiT': (0.264, 0.023),
                 'AgentSim': (0.375, 0.026),
                 'SIMOD': (0.279, 0.015),
                 'RIMS': (0.221, 0.02),
                 'DSIM': (0.266, 0.001)},
         'CVS': {'ProSiT': (0.016, 0.001),
                 'SIMOD': (0.021, 0.002),
                 'RIMS': (0.221, 0.003),
                 'DSIM': (0.256, 0.002)},
         'Production': {'ProSiT': (0.486, 0.029),
                        'AgentSim': (0.561, 0.023),
                        'SIMOD': (0.565, 0.021),
                        'RIMS': (0.764, 0.015),
                        'DSIM': (0.763, 0.017)},
         'PurchasingExample': {'ProSiT': (0.248, 0.014),
                               'AgentSim': (0.42, 0.011),
                               'SIMOD': (0.253, 0.024),
                               'RIMS': (0.16, 0.016),
                               'DSIM': (0.439, 0.005)},
         'SynLoan': {'ProSiT': (0.127, 0.012),
                     'SIMOD': (0.113, 0.007),
                     'RIMS': (0.4, 0.018),
                     'DSIM': (0.439, 0.001)}},
 'car': {'BPIC12W': {'ProSiT': (12.519, 6.272),
                     'AgentSim': (51.408, 0.136),
                     'SIMOD': (14.35, 4.883),
                     'RIMS': (24.446, 4.754),
                     'DSIM': (23.802, 5.022)},
         'BPIC17W': {'ProSiT': (141.054, 11.784),
                     'AgentSim': (178.596, 0.128),
                     'SIMOD': (156.445, 7.511),
                     'RIMS': (39.051, 9.829),
                     'DSIM': (34.904, 10.032)},
         'P2P-1000': {'ProSiT': (71.011, 16.839),
                      'SIMOD': (66.852, 30.26),
                      'RIMS': (235.137, 9.745),
                      'DSIM': (236.599, 7.354)},
         'P2P-2000': {'ProSiT': (132.228, 50.844),
                      'SIMOD': (49.223, 19.658),
                      'RIMS': (646.689, 3.856),
                      'DSIM': (641.835, 5.488)},
         'ACR': {'ProSiT': (252.393, 44.541),
                 'AgentSim': (368.133, 0.037),
                 'SIMOD': (292.398, 26.28),
                 'RIMS': (244.998, 6.64),
                 'DSIM': (239.763, 8.422)},
         'CVS': {'ProSiT': (5.649, 0.772),
                 'SIMOD': (5.184, 0.9),
                 'RIMS': (19.287, 1.336),
                 'DSIM': (20.37, 1.751)},
         'Production': {'ProSiT': (32.029, 13.758),
                        'AgentSim': (95.884, 0.591),
                        'SIMOD': (69.858, 28.931),
                        'RIMS': (167.611, 3.652),
                        'DSIM': (167.733, 2.623)},
         'PurchasingExample': {'ProSiT': (633.149, 48.735),
                               'AgentSim': (787.604, 0.0),
                               'SIMOD': (594.018, 34.727),
                               'RIMS': (900.444, 5.919),
                               'DSIM': (893.916, 9.187)},
         'SynLoan': {'ProSiT': (3.707, 1.978),
                     'SIMOD': (3.498, 2.216),
                     'RIMS': (326.658, 3.657),
                     'DSIM': (326.035, 3.963)}},
 'ctd': {'BPIC12W': {'ProSiT': (44.553, 3.902),
                     'AgentSim': (70.855, 3.172),
                     'SIMOD': (77.236, 2.594),
                     'RIMS': (100.291, 2.747),
                     'DSIM': (163.183, 1.692)},
         'BPIC17W': {'ProSiT': (106.389, 2.153),
                     'AgentSim': (67.475, 2.66),
                     'SIMOD': (46.082, 1.539),
                     'RIMS': (87.0, 0.948),
                     'DSIM': (115.763, 0.723)},
         'P2P-1000': {'ProSiT': (14.85, 1.829),
                      'SIMOD': (41.174, 12.886),
                      'RIMS': (9.249, 0.256),
                      'DSIM': (10.052, 0.281)},
         'P2P-2000': {'ProSiT': (10.629, 1.726),
                      'SIMOD': (78.019, 14.557),
                      'RIMS': (2.483, 1.483),
                      'DSIM': (2.369, 0.631)},
         'ACR': {'ProSiT': (229.489, 18.183),
                 'AgentSim': (470.875, 45.046),
                 'SIMOD': (355.149, 28.821),
                 'RIMS': (46.61, 7.25),
                 'DSIM': (64.036, 1.49)},
         'CVS': {'ProSiT': (145.686, 0.758),
                 'SIMOD': (103.121, 1.681),
                 'RIMS': (97.901, 1.064),
                 'DSIM': (37.627, 0.907)},
         'Production': {'ProSiT': (335.896, 49.422),
                        'AgentSim': (221.173, 29.017),
                        'SIMOD': (410.071, 79.595),
                        'RIMS': (34.816, 6.33),
                        'DSIM': (21.556, 4.58)},
         'PurchasingExample': {'ProSiT': (402.283, 25.978),
                               'AgentSim': (448.43, 15.507),
                               'SIMOD': (445.978, 22.374),
                               'RIMS': (584.773, 8.558),
                               'DSIM': (632.178, 4.503)},
         'SynLoan': {'ProSiT': (153.435, 12.224),
                     'SIMOD': (457.96, 27.6),
                     'RIMS': (290.938, 16.227),
                     'DSIM': (467.776, 20.804)}},
 'etd_entropy': {'BPIC12W': {'ProSiT': (2.863, 0.014),
                             'AgentSim': (2.376, 0.035),
                             'SIMOD': (2.208, 0.01),
                             'RIMS': (0.298, 0.006),
                             'DSIM': (0.3, 0.011)},
                 'BPIC17W': {'ProSiT': (2.618, 0.007),
                             'AgentSim': (2.804, 0.03),
                             'SIMOD': (2.108, 0.009),
                             'RIMS': (0.688, 0.006),
                             'DSIM': (0.512, 0.004)},
                 'P2P-1000': {'ProSiT': (1.615, 0.017),
                              'SIMOD': (0.762, 0.028),
                              'RIMS': (0.296, 0.017),
                              'DSIM': (0.347, 0.029)},
                 'P2P-2000': {'ProSiT': (1.872, 0.009),
                              'SIMOD': (0.655, 0.017),
                              'RIMS': (0.271, 0.034),
                              'DSIM': (0.494, 0.025)},
                 'ACR': {'ProSiT': (0.943, 0.021),
                         'AgentSim': (0.369, 0.061),
                         'SIMOD': (0.712, 0.032),
                         'RIMS': (0.873, 0.019),
                         'DSIM': (0.786, 0.045)},
                 'CVS': {'ProSiT': (0.094, 0.002),
                         'SIMOD': (0.242, 0.003),
                         'RIMS': (0.145, 0.004),
                         'DSIM': (0.056, 0.002)},
                 'Production': {'ProSiT': (1.641, 0.044),
                                'AgentSim': (1.009, 0.083),
                                'SIMOD': (1.287, 0.038),
                                'RIMS': (0.914, 0.039),
                                'DSIM': (1.27, 0.05)},
                 'PurchasingExample': {'ProSiT': (1.46, 0.008),
                                       'AgentSim': (0.257, 0.012),
                                       'SIMOD': (1.102, 0.064),
                                       'RIMS': (0.334, 0.027),
                                       'DSIM': (0.595, 0.047)},
                 'SynLoan': {'ProSiT': (2.304, 0.009),
                             'SIMOD': (2.26, 0.008),
                             'RIMS': (1.057, 0.014),
                             'DSIM': (0.343, 0.011)}},
 '2rgd': {'BPIC12W': {'ProSiT': (0.632, 0.006),
                      'AgentSim': (0.694, 0.021),
                      'SIMOD': (0.692, 0.004)},
          'BPIC17W': {'ProSiT': (0.638, 0.003),
                      'AgentSim': (0.827, 0.007),
                      'SIMOD': (0.777, 0.001)},
          'P2P-1000': {'ProSiT': (0.134, 0.007), 'SIMOD': (0.188, 0.01)},
          'P2P-2000': {'ProSiT': (0.126, 0.009), 'SIMOD': (0.216, 0.012)},
          'ACR': {'ProSiT': (0.835, 0.009), 'AgentSim': (0.969, 0.003), 'SIMOD': (0.962, 0.004)},
          'CVS': {'ProSiT': (0.016, 0.003), 'SIMOD': (0.368, 0.002)},
          'Production': {'ProSiT': (0.624, 0.014),
                         'AgentSim': (0.845, 0.018),
                         'SIMOD': (0.833, 0.011)},
          'PurchasingExample': {'ProSiT': (0.379, 0.011),
                                'AgentSim': (0.563, 0.019),
                                'SIMOD': (0.396, 0.015)},
          'SynLoan': {'ProSiT': (0.092, 0.005), 'SIMOD': (0.087, 0.006)}},
 'arrival_tree': {'BPIC12W': {'ProSiT': (1.569, 0.192),
                              'AgentSim': (6.647, 0.356),
                              'SIMOD': (7.021, 2.302),
                              'RIMS': (6.347, 0.629),
                              'DSIM': (6.873, 0.608)},
                  'BPIC17W': {'ProSiT': (0.609, 0.04),
                              'AgentSim': (3.852, 0.374),
                              'SIMOD': (7.636, 0.332),
                              'RIMS': (3.321, 0.164),
                              'DSIM': (3.336, 0.127)},
                  'P2P-1000': {'ProSiT': (10.964, 4.279),
                               'SIMOD': (33.912, 4.902),
                               'RIMS': (38.388, 1.78),
                               'DSIM': (39.166, 1.086)},
                  'P2P-2000': {'ProSiT': (14.482, 2.454),
                               'SIMOD': (92.311, 2.829),
                               'RIMS': (60.282, 0.925),
                               'DSIM': (59.383, 0.73)},
                  'ACR': {'ProSiT': (24.933, 5.863),
                          'AgentSim': (62.14, 1.632),
                          'SIMOD': (87.863, 12.657),
                          'RIMS': (64.29, 3.194),
                          'DSIM': (61.118, 3.35)},
                  'CVS': {'ProSiT': (0.273, 0.017),
                          'SIMOD': (0.279, 0.01),
                          'RIMS': (0.605, 0.034),
                          'DSIM': (0.593, 0.047)},
                  'Production': {'ProSiT': (59.243, 12.987),
                                 'AgentSim': (112.604, 2.955),
                                 'SIMOD': (358.573, 296.214),
                                 'RIMS': (195.779, 4.119),
                                 'DSIM': (194.773, 5.127)},
                  'PurchasingExample': {'ProSiT': (52.333, 14.097),
                                        'AgentSim': (113.895, 0.0),
                                        'SIMOD': (52.75, 14.503),
                                        'RIMS': (200.353, 3.158),
                                        'DSIM': (196.498, 4.706)},
                  'SynLoan': {'ProSiT': (2.657, 0.677),
                              'SIMOD': (12.201, 1.305),
                              'RIMS': (98.475, 1.006),
                              'DSIM': (98.44, 0.993)}},
 'execution_tree': {'BPIC12W': {'ProSiT': (3.676, 0.343), 'SIMOD': (26.675, 1.39)},
                    'BPIC17W': {'ProSiT': (1.384, 0.088), 'SIMOD': (11.159, 1.683)},
                    'P2P-1000': {'ProSiT': (4.474, 0.496), 'SIMOD': (23.782, 2.889)},
                    'P2P-2000': {'ProSiT': (4.031, 0.323), 'SIMOD': (31.343, 2.202)},
                    'ACR': {'ProSiT': (13.075, 1.846),
                            'AgentSim': (151.73, 17.644),
                            'SIMOD': (90.974, 14.517)},
                    'CVS': {'ProSiT': (0.492, 0.075), 'SIMOD': (1.185, 0.219)},
                    'Production': {'ProSiT': (33.297, 6.677), 'SIMOD': (119.619, 29.23)},
                    'PurchasingExample': {'ProSiT': (7.732, 2.187),
                                          'AgentSim': (144.173, 8.625),
                                          'SIMOD': (66.646, 16.199)},
                    'SynLoan': {'ProSiT': (2.629, 0.165), 'SIMOD': (6.042, 0.25)}},
 'waiting_tree': {'BPIC12W': {'ProSiT': (1152.414, 195.068), 'SIMOD': (1960.119, 15.59)},
                  'BPIC17W': {'ProSiT': (804.995, 101.693), 'SIMOD': (2366.841, 24.195)},
                  'P2P-1000': {'ProSiT': (14.246, 1.557), 'SIMOD': (75.743, 9.62)},
                  'P2P-2000': {'ProSiT': (8.604, 1.389), 'SIMOD': (228.306, 21.74)},
                  'ACR': {'ProSiT': (1125.166, 215.739),
                          'AgentSim': (4765.251, 601.952),
                          'SIMOD': (3438.467, 176.898)},
                  'CVS': {'ProSiT': (28.47, 1.328), 'SIMOD': (540.444, 4.188)},
                  'Production': {'ProSiT': (290.748, 60.482), 'SIMOD': (1694.41, 243.01)},
                  'PurchasingExample': {'ProSiT': (307.925, 83.452),
                                        'AgentSim': (436.923, 39.425),
                                        'SIMOD': (439.847, 22.865)},
                  'SynLoan': {'ProSiT': (313.554, 55.754), 'SIMOD': (575.26, 33.825)}},
 'transition_tree': {'BPIC12W': {'ProSiT': (0.012, 0.003),
                                 'SIMOD': (0.109, 0.004),
                                 'RIMS': (0.114, 0.004)},
                     'BPIC17W': {'ProSiT': (0.005, 0.0),
                                 'SIMOD': (0.115, 0.001),
                                 'RIMS': (0.136, 0.002)},
                     'P2P-1000': {'ProSiT': (0.006, 0.002),
                                  'SIMOD': (0.03, 0.002),
                                  'RIMS': (0.027, 0.003),
                                  'DSIM': (0.028, 0.003)},
                     'P2P-2000': {'ProSiT': (0.004, 0.001),
                                  'SIMOD': (0.026, 0.002),
                                  'RIMS': (0.027, 0.002),
                                  'DSIM': (0.025, 0.002)},
                     'ACR': {'ProSiT': (0.021, 0.006),
                             'AgentSim': (0.044, 0.009),
                             'SIMOD': (0.086, 0.014),
                             'RIMS': (0.13, 0.014),
                             'DSIM': (0.292, 0.054)},
                     'CVS': {'ProSiT': (0.002, 0.001),
                             'SIMOD': (0.038, 0.001),
                             'RIMS': (0.178, 0.002),
                             'DSIM': (0.081, 0.001)},
                     'Production': {'ProSiT': (0.035, 0.008), 'SIMOD': (0.072, 0.005)},
                     'PurchasingExample': {'ProSiT': (0.012, 0.003),
                                           'AgentSim': (0.042, 0.008),
                                           'SIMOD': (0.034, 0.007),
                                           'RIMS': (0.153, 0.008),
                                           'DSIM': (0.083, 0.004)},
                     'SynLoan': {'ProSiT': (0.008, 0.002),
                                 'SIMOD': (0.041, 0.003),
                                 'RIMS': (0.129, 0.004)}},
 'resource_tree': {'BPIC12W': {'ProSiT': (0.006, 0.001), 'SIMOD': (0.044, 0.0)},
                   'BPIC17W': {'ProSiT': (0.005, 0.0), 'SIMOD': (0.015, 0.0)},
                   'P2P-1000': {'ProSiT': (0.101, 0.004), 'SIMOD': (0.113, 0.008)},
                   'P2P-2000': {'ProSiT': (0.126, 0.006), 'SIMOD': (0.209, 0.008)},
                   'ACR': {'ProSiT': (0.005, 0.0),
                           'AgentSim': (0.01, 0.001),
                           'SIMOD': (0.011, 0.001)},
                   'CVS': {'ProSiT': (0.006, 0.001), 'SIMOD': (0.166, 0.002)},
                   'Production': {'ProSiT': (0.037, 0.006), 'SIMOD': (0.149, 0.017)},
                   'PurchasingExample': {'ProSiT': (0.021, 0.002),
                                         'AgentSim': (0.115, 0.006),
                                         'SIMOD': (0.023, 0.002)},
                   'SynLoan': {'ProSiT': (0.072, 0.004), 'SIMOD': (0.051, 0.004)}}}

In [34]:
# formatting order: dataset -> method -> metric -> mean/std

# baseline = {}
# for metric_key, metric_value in TABLES.items():
#     for dataset_key, dataset_value in metric_value.items():
#         for method_key, method_value in dataset_value.items():
#             baseline.setdefault(dataset_key, {}).setdefault(method_key, {}).setdefault(metric_key, {})
#             baseline[dataset_key][method_key][metric_key]['mean'] = method_value[0]
#             baseline[dataset_key][method_key][metric_key]['std'] = method_value[1]

baseline = load_json_file('results_github/summary.json')

for dataset_key, dataset_value in baseline.items():
    # print(f"Dataset: {dataset_key}")

    del dataset_value['agentsimulator']  # Remove 'agentsimulator' key if it exists

    # print(f"  Methods: {', '.join(dataset_value.keys())}")  # Print the remaining method keys
    # print(f"  Metrics: {', '.join(dataset_value[next(iter(dataset_value))].keys())}")  # Print the metrics keys from the first method
    # print()
    
    baseline[dataset_key] = dataset_value  # Update the summary with the modified dataset_value

# Remove datasets and methods that are not in the specified list
dataset_keys, dataset_values = zip(*baseline.items())
for dataset_key, dataset_value in zip(dataset_keys, dataset_values):
    if dataset_key not in ['BPI_Challenge_2012_W_Two_TS', 'BPI_Challenge_2017_W_Two_TS', 'confidential_2000', 'ConsultaDataMining201618', 'cvs_pharmacy']:
        del baseline[dataset_key]  # Remove datasets that are not in the specified list
# dataset_keys, dataset_values = zip(*baseline.items())
# for dataset_key, dataset_value in zip(dataset_keys, dataset_values):
#     method_keys, method_values = zip(*dataset_value.items())
#     for method_key, method_value in zip(method_keys, method_values):
#         if method_key not in ['ProSiT', 'SIMOD', 'RIMS', 'DSIM']:
#             del baseline[dataset_key][method_key]  # Remove methods that are not in the specified list

print_json_structure(baseline)  # Print the structure of the baseline dictionary

save_json_file(baseline, 'results/baseline_modified.json')  # Save the modified summary back to the JSON file



Dataset: BPI_Challenge_2012_W_Two_TS
  Methods: prosit, simod, rims, dsim
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree

Dataset: BPI_Challenge_2017_W_Two_TS
  Methods: prosit, simod, rims, dsim
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree

Dataset: confidential_2000
  Methods: prosit, simod, rims, dsim
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree

Dataset: ConsultaDataMining201618
  Methods: prosit, simod, rims, dsim
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree

Dataset: cvs_pharmacy
  Methods: prosit, simod, rims, dsim
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tree, arrival_tree, car, resource_tree



In [35]:
# same keys for both baseline and summary
key_convert = {
    'BPI_Challenge_2012_W_Two_TS': 'BPIC12W',
    'BPI_Challenge_2017_W_Two_TS': 'BPIC17W',
    'cvs_pharmacy': 'CVS',
    'ConsultaDataMining201618': 'ACR',
    'confidential_2000': 'P2P',
    'BPIC12W': 'BPIC12W',
    'BPIC17W': 'BPIC17W',
    'CVS': 'CVS',
    'ACR': 'ACR',
    'P2P-2000': 'P2P',
    'prosit': 'prosit',
    'simod': 'simod',
    'rims': 'rims',
    'dsim': 'dsim',
    'ProSiT': 'prosit',
    'SIMOD': 'simod',
    'RIMS': 'rims',
    'DSIM': 'dsim',
}
summary = load_json_file('results/summary_modified.json')  # Load the modified summary JSON file
summary_renamed = {}
for dataset_key, dataset_value in summary.items():
    summary_renamed[key_convert[dataset_key]] = {}
    for method_key, method_value in dataset_value.items():
        summary_renamed[key_convert[dataset_key]][key_convert[method_key]] = {}
        for metric_key, metric_value in method_value.items():
            summary_renamed[key_convert[dataset_key]][key_convert[method_key]][metric_key] = metric_value

print_json_structure(summary_renamed)  # Print the structure of the renamed summary dictionary
save_json_file(summary_renamed, 'results/summary_mod.json')  # Save the renamed summary to a new JSON file

tables_renamed = {}
for dataset_key, dataset_value in baseline.items():
    tables_renamed[key_convert[dataset_key]] = {}
    for method_key, method_value in dataset_value.items():
        # if method_key not in ['ProSiT', 'SIMOD', 'RIMS', 'DSIM']:
        tables_renamed[key_convert[dataset_key]][key_convert[method_key]] = {}
        for metric_key, metric_value in method_value.items():
            tables_renamed[key_convert[dataset_key]][key_convert[method_key]][metric_key] = metric_value

print_json_structure(tables_renamed)  # Print the structure of the renamed tables dictionary
save_json_file(tables_renamed, 'results/baseline_modified.json')  # Save the renamed tables to a new JSON file

Dataset: BPIC12W
  Methods: prosit, simod, rims, dsim
  Metrics: transition_tree, resource_tree, 2gd, ctd, car, arrival_tree, waiting_tree, execution_tree, etd_entropy, 2rgd

Dataset: BPIC17W
  Methods: prosit, simod, rims, dsim
  Metrics: transition_tree, resource_tree, 2gd, ctd, car, arrival_tree, waiting_tree, execution_tree, etd_entropy, 2rgd

Dataset: ACR
  Methods: prosit, simod, rims, dsim
  Metrics: transition_tree, resource_tree, 2gd, ctd, car, arrival_tree, waiting_tree, execution_tree, etd_entropy, 2rgd

Dataset: CVS
  Methods: prosit, simod, rims, dsim
  Metrics: car, 2gd, transition_tree, resource_tree, waiting_tree, arrival_tree, etd_entropy, ctd, execution_tree, 2rgd

Dataset: P2P
  Methods: prosit, simod, rims, dsim
  Metrics: car, 2gd, transition_tree, resource_tree, waiting_tree, arrival_tree, etd_entropy, ctd, execution_tree, 2rgd

Dataset: BPIC12W
  Methods: prosit, simod, rims, dsim
  Metrics: execution_tree, 2gd, ctd, waiting_tree, etd_entropy, 2rgd, transition_tr

# perform tests

In [36]:
#t-tests

#           ngd    ctd    car    -----   entropy        Rule-ATD        Rule-CFD           Rule-ETD          -------          Rule-WTD
# metrics = ['2gd', 'ctd', 'car', '2rgd', 'etd_entropy', 'arrival_tree', 'transition_tree', 'execution_tree', 'resource_tree', 'waiting_tree']
metrics = ['2gd', 'ctd', 'car', 'etd_entropy', 'arrival_tree', 'transition_tree', 'execution_tree', 'waiting_tree']
datasets = ['BPIC12W', 'BPIC17W', 'ACR', 'CVS', 'P2P']
methods = ['prosit', 'simod', 'rims', 'dsim']

print(len(metrics)*len(datasets)*len(methods))

summary = load_json_file('results/summary_mod.json')
baseline = load_json_file('results/baseline_modified.json')

count_rejected = 0
count_tested = 0
p_values = []
for dataset_key, dataset_value in summary.items():
    for method_key, method_value in dataset_value.items():
        for metric_key, metric_value in method_value.items():
            if metric_key in metrics:
                print(f"Dataset: {dataset_key},     Method: {method_key},       Metric: {metric_key}")
                t_statistic = stats.ttest_ind_from_stats(
                    mean1=metric_value['mean'],
                    std1=metric_value['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                )
                count_tested += 1
                print(t_statistic)
                p_values.append(t_statistic.pvalue)
                if t_statistic.pvalue < 0.05:
                    print("Reject the null hypothesis: The means are significantly different.")
                    count_rejected += 1
                print()

print(f'Number of tests performed: {count_tested}')
print(f'Number of rejections: {count_rejected}')




160
Dataset: BPIC12W,     Method: prosit,       Metric: transition_tree
Ttest_indResult(statistic=-1.035223251372286, pvalue=0.314556339252137)

Dataset: BPIC12W,     Method: prosit,       Metric: 2gd
Ttest_indResult(statistic=-0.36141212360969766, pvalue=0.723432824971096)

Dataset: BPIC12W,     Method: prosit,       Metric: ctd
Ttest_indResult(statistic=11.760359997070932, pvalue=1.1200869126799302e-09)
Reject the null hypothesis: The means are significantly different.

Dataset: BPIC12W,     Method: prosit,       Metric: car
Ttest_indResult(statistic=4.241859834651385, pvalue=0.0006512103397445815)
Reject the null hypothesis: The means are significantly different.

Dataset: BPIC12W,     Method: prosit,       Metric: arrival_tree
Ttest_indResult(statistic=-11.587110143731532, pvalue=8.879248827179007e-10)
Reject the null hypothesis: The means are significantly different.

Dataset: BPIC12W,     Method: prosit,       Metric: waiting_tree
Ttest_indResult(statistic=-4.23997001009068, pval

In [37]:
combined_statistic, combined_p_value = stats.combine_pvalues(p_values, method='fisher')
print(f'Combined p-value (Fisher): {combined_p_value}')
if combined_p_value < 0.05:
    print("Reject the null hypothesis: The means are significantly different.")


holms_corrected = multitest.multipletests(p_values, alpha=0.05, method='holm')
print(holms_corrected[0])
num_rejections_holm = sum(holms_corrected[0])
print(f'Number of rejections after Holm-Bonferroni correction: {num_rejections_holm}')

Combined p-value (Fisher): 6.99652921812569e-98
Reject the null hypothesis: The means are significantly different.
[False False  True False  True False  True  True False False False False
 False False False False False False False False  True False False False
  True  True  True  True  True False False False False False False False
 False False False False False False False False False False False False
 False False False False  True False False False False False  True False
 False False False False False False  True False False False False False
 False  True False False False False False False False False  True False
 False False False False False False False False False False False False
 False False False False False False  True  True False  True False False
  True False False False False False False False  True False False False
 False False  True False False False]
Number of rejections after Holm-Bonferroni correction: 21


In [38]:
# table columns: Dataset, Method, Metric, Mean (Summary), Std (Summary), Mean (Baseline), Std (Baseline), t-statistic, p-value, holms p-value, Holm Reject Null Hypothesis
results_table = []
for dataset_key, dataset_value in summary.items():
    for method_key, method_value in dataset_value.items():
        for metric_key, metric_value in method_value.items():
            if metric_key in metrics:
                t_statistic = stats.ttest_ind_from_stats(
                    mean1=metric_value['mean'],
                    std1=metric_value['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                )
                holm_reject_null = holms_corrected[0][count_tested - len(p_values) + p_values.index(t_statistic.pvalue)]
                results_table.append({
                    'Dataset': dataset_key,
                    'Method': method_key,
                    'Metric': metric_key,
                    'Mean (Summary)': metric_value['mean'],
                    'Std (Summary)': metric_value['std'],
                    'Mean (Baseline)': baseline[dataset_key][method_key][metric_key]['mean'],
                    'Std (Baseline)': baseline[dataset_key][method_key][metric_key]['std'],
                    't-statistic': t_statistic.statistic,
                    'p-value': t_statistic.pvalue,
                    'Holm p-value': holms_corrected[1][count_tested - len(p_values) + p_values.index(t_statistic.pvalue)],
                    'Holm Reject Null Hypothesis': holm_reject_null
                })


In [39]:
print(results_table)

[{'Dataset': 'BPIC12W', 'Method': 'prosit', 'Metric': 'transition_tree', 'Mean (Summary)': 0.010430418425225439, 'Std (Summary)': 0.003086663504500118, 'Mean (Baseline)': 0.011766023376553263, 'Std (Baseline)': 0.002667896100051384, 't-statistic': -1.035223251372286, 'p-value': 0.314556339252137, 'Holm p-value': 1.0, 'Holm Reject Null Hypothesis': False}, {'Dataset': 'BPIC12W', 'Method': 'prosit', 'Metric': '2gd', 'Mean (Summary)': 0.3974144221502919, 'Std (Summary)': 0.00816596715288391, 'Mean (Baseline)': 0.39846203020643667, 'Std (Baseline)': 0.004163991793604506, 't-statistic': -0.36141212360969766, 'p-value': 0.723432824971096, 'Holm p-value': 1.0, 'Holm Reject Null Hypothesis': False}, {'Dataset': 'BPIC12W', 'Method': 'prosit', 'Metric': 'ctd', 'Mean (Summary)': 67.52791666666668, 'Std (Summary)': 4.7892784374463835, 'Mean (Baseline)': 44.553250000000006, 'Std (Baseline)': 3.9021921625685065, 't-statistic': 11.760359997070932, 'p-value': 1.1200869126799302e-09, 'Holm p-value': 1.

# results table of means

In [12]:
print("""\\begin{table}[htbp]
\\centering
\\begin{tabular}{ll*{7}{>{\\centering\\arraybackslash}p{1.2cm}}>{\\centering\\arraybackslash}p{1.6cm}}
\\toprule
\\textbf{Method} & \\textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\\\""")
for dataset_key, dataset_value in summary.items():
    print("\\midrule")
    print(f"\\multicolumn{{9}}{{l}}{{{dataset_key}}}\\\\")
    for method_key, method_value in summary[dataset_key].items():
        lineprint_our = method_key + " & our"
        lineprint_baseline = method_key + " & paper"
        for metric_key in ['2gd', 'ctd', 'car', 'transition_tree', 'arrival_tree', 'execution_tree', 'waiting_tree', 'etd_entropy']:
            if metric_key in method_value and metric_key in baseline[dataset_key][method_key]:
                lineprint_our += f" & {method_value[metric_key]['mean']:.2f}"
                lineprint_baseline += f" & {baseline[dataset_key][method_key][metric_key]['mean']:.2f}"
                if holms_corrected[0][count_tested - len(p_values) + p_values.index(stats.ttest_ind_from_stats(
                    mean1=method_value[metric_key]['mean'],
                    std1=method_value[metric_key]['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                ).pvalue)]:
                    lineprint_our += " \\cellcolor{red!20}"
                    lineprint_baseline += " \\cellcolor{red!20}"
            else:
                lineprint_our += " & N/A"
                lineprint_baseline += " & N/A"
        print(lineprint_our + " \\\\")
        print(lineprint_baseline + " \\\\")
print("""\\bottomrule
\end{tabular}
\caption{Metric results of the author's simulation compared to its posted results. Cells highlighted in red indicate results are significantly different from the baseline (p < 0.05) after Holm-Bonferroni correction. N/A's are placed where no data was collected by the author, or the author's replication code was unable to produce results.}
\label{tab:results}
\end{table}""")

\begin{table}[htbp]
\centering
\begin{tabular}{ll*{7}{>{\centering\arraybackslash}p{1.2cm}}>{\centering\arraybackslash}p{1.6cm}}
\toprule
\textbf{Method} & \textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\
\midrule
\multicolumn{9}{l}{BPIC12W}\\
prosit & our & 0.40 & 67.53 \cellcolor{red!20} & 27.73 & 0.01 & 0.58 \cellcolor{red!20} & 1.69 \cellcolor{red!20} & 857.32 & 2.58 \cellcolor{red!20} \\
prosit & paper & 0.40 & 44.55 \cellcolor{red!20} & 12.52 & 0.01 & 1.57 \cellcolor{red!20} & 3.68 \cellcolor{red!20} & 1152.41 & 2.86 \cellcolor{red!20} \\
simod & our & 0.52 & 77.24 & 14.35 & 0.11 & 3.48 & 25.45 & 1973.60 & 2.21 \\
simod & paper & 0.53 & 77.24 & 14.35 & 0.11 & 7.02 & 26.68 & 1960.12 & 2.21 \\
rims & our & N/A & N/A & N/A & N/A & N/A & N/A & N/A & N/A \\
rims & paper & N/A & N/A & N/A & N/A & N/A & N/A & N/A & N/A \\
dsim & our & 0.51 & 163.18 & 23.80 & N/A & 1.52 \cellcolor{red!20} & N/A & N/A & 0.30 \\
dsim & paper & 0.51 &

In [40]:
# 
print("""\\begin{table}[htbp]
\\centering\\scriptsize
\\begin{tabular}{ll*{7}{>{\\centering\\arraybackslash}p{1.2cm}}>{\\centering\\arraybackslash}p{1.6cm}}
\\toprule
\\textbf{Method} & \\textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\\\""")
for dataset_key, dataset_value in summary.items():
    print("\\midrule")
    print(f"\\multicolumn{{9}}{{l}}{{{dataset_key}}}\\\\")
    for method_key, method_value in summary[dataset_key].items():
        lineprint_our = method_key + " & our"
        lineprint_baseline = method_key + " & paper"
        for metric_key in ['2gd', 'ctd', 'car', 'transition_tree', 'arrival_tree', 'execution_tree', 'waiting_tree', 'etd_entropy']:
            if metric_key in method_value and metric_key in baseline[dataset_key][method_key]:
                lineprint_our += f" & {method_value[metric_key]['mean']:.2f}"+"{\\tiny$\\pm$"+f"{method_value[metric_key]['std']:.2f}"+"}"
                lineprint_baseline += f" & {baseline[dataset_key][method_key][metric_key]['mean']:.2f}"+"{\\tiny$\\pm$"+f"{baseline[dataset_key][method_key][metric_key]['std']:.2f}"+"}"
                if holms_corrected[0][count_tested - len(p_values) + p_values.index(stats.ttest_ind_from_stats(
                    mean1=method_value[metric_key]['mean'],
                    std1=method_value[metric_key]['std'],
                    nobs1=10,
                    mean2=baseline[dataset_key][method_key][metric_key]['mean'],
                    std2=baseline[dataset_key][method_key][metric_key]['std'],
                    nobs2=10,
                    equal_var=False
                ).pvalue)]:
                    lineprint_our += " \\cellcolor{red!20}"
                    lineprint_baseline += " \\cellcolor{red!20}"
            else:
                lineprint_our += " & N/A"
                lineprint_baseline += " & N/A"
        print(lineprint_our + " \\\\")
        print(lineprint_baseline + " \\\\")
print("""\\bottomrule
\end{tabular}
\caption{Metric results of the author's simulation compared to its posted results. Cells highlighted in red indicate results are significantly different from the baseline (p < 0.05) after Holm-Bonferroni correction. N/A's are placed where no data was collected by the author, or the author's replication code was unable to produce results.}
\label{tab:results}
\end{table}""")

\begin{table}[htbp]
\centering\scriptsize
\begin{tabular}{ll*{7}{>{\centering\arraybackslash}p{1.2cm}}>{\centering\arraybackslash}p{1.6cm}}
\toprule
\textbf{Method} & \textbf{Experiment} & NGD & CTD & CAR & Rule-CFD & Rule-ATD & Rule-ETD & Rule-WTD & Exec. Time Entropy \\
\midrule
\multicolumn{9}{l}{BPIC12W}\\
prosit & our & 0.40{\tiny$\pm$0.01} & 67.53{\tiny$\pm$4.79} \cellcolor{red!20} & 27.73{\tiny$\pm$9.45} & 0.01{\tiny$\pm$0.00} & 0.58{\tiny$\pm$0.19} \cellcolor{red!20} & 1.69{\tiny$\pm$0.77} \cellcolor{red!20} & 857.32{\tiny$\pm$101.92} & 2.58{\tiny$\pm$0.01} \cellcolor{red!20} \\
prosit & paper & 0.40{\tiny$\pm$0.00} & 44.55{\tiny$\pm$3.90} \cellcolor{red!20} & 12.52{\tiny$\pm$6.27} & 0.01{\tiny$\pm$0.00} & 1.57{\tiny$\pm$0.19} \cellcolor{red!20} & 3.68{\tiny$\pm$0.34} \cellcolor{red!20} & 1152.41{\tiny$\pm$195.07} & 2.86{\tiny$\pm$0.01} \cellcolor{red!20} \\
simod & our & 0.52{\tiny$\pm$0.01} & 77.24{\tiny$\pm$2.59} & 14.35{\tiny$\pm$4.88} & 0.11{\tiny$\pm$0.00} & 3.48{\tiny$\p